Notebook for creating an indexing pipeline for RAG model.

Step 1:
Create a structured and cleaned dataset.

In [ ]:
import json

In [52]:
path_raw = "../recipe_crawler/recipes.json"
with open(path_raw, 'r') as f:
    recipes = json.load(f)

# print the first few recipes
print(json.dumps(recipes[0:5], indent=4))

[
    {
        "recipe_name": "Nagi\u2019s Salad Sandwich",
        "ingredient_groups": [
            {
                "group_name": null,
                "ingredients": [
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "soft bread roll",
                        "notes": ", wholemeal or white (or sliced bread, wraps)"
                    },
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "small carrot",
                        "notes": ", peeled, grated using a box grater (3/4 cup lightly packed per sandwich)"
                    },
                    {
                        "amount": "1/2",
                        "unit": "",
                        "name": "ripe avocado",
                        "notes": ""
                    },
                    {
                        "amount": "1",
                        "unit

Cleaning to do: (more might pop up)
- Strip leading ', ' and ' ' from notes
- Normalise group names that are null to Main
- Move unit-like tokens from amount to unit
- Separate amounts that contain amount and units. And hanlde dual amounts by adding alt field where necessary
- Check if for missing amount and units, they are included in the notes instead
- Normalise group names in instructions

In [ ]:
# function to normalise group names
def normalise_group_name(group_name):
    if group_name == None:
        return "Main"
    else:

        # this part is also done by a function later below called clean_group_name
        new_name = group_name.rstrip(":").strip()
        new_name = new_name.title()
        return new_name

# normalse group names and remove leading commaas and spaces from notes
with open(path_raw, 'r') as f:
    recipes = json.load(f)

for recipe in recipes:
    for group in recipe["ingredient_groups"]:
        group["group_name"] = normalise_group_name(group["group_name"])
        for ing in group["ingredients"]:
            # remove leading commas or spaces from notes
            if ing["notes"]:
                ing["notes"] = ing["notes"].lstrip(",")
                ing["notes"] = ing["notes"].lstrip(" ")

# write to new file
with open("recipes_cleaned.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)
            

In [54]:
# verify 
clean_path = "recipes_cleaned.json"
with open(clean_path, 'r') as f:
    recipes = json.load(f)

# print the first few recipes
for recipe in recipes[:10]:
    for group in recipe["ingredient_groups"]:
        print(group["group_name"])

print("------------------ ")
print(json.dumps(recipes[9], indent=4))

Main
Dressing
Main
Egg Wash
Main
Serving
Tuna
Lemon Vinaigrette
Pickled Onion
Crostinis
Main
Cheese Sauce (Mornay Sauce)
Chocolate Base
Chocolate Frosting
3 Layer Vanilla Cake
My Favourite Chocolate Frosting
Main
Main
Herbs Butter
Potatoes
Beef Filling
Beef Spice Mix
Corn Layer
Assembling / Baking
------------------ 
{
    "recipe_name": "Pastel de Choclo (Chilean Beef and Corn Casserole)",
    "ingredient_groups": [
        {
            "group_name": "Beef Filling",
            "ingredients": [
                {
                    "amount": "2",
                    "unit": "tbsp",
                    "name": "olive oil",
                    "notes": ""
                },
                {
                    "amount": "1 kg / 2 lb",
                    "unit": "",
                    "name": "beef mince",
                    "notes": "(ground beef)"
                },
                {
                    "amount": "3",
                    "unit": "",
                    "name": "la

Note that some group names in ingredients are not describing the component of the dish itself (e.g. sauce, dressing, etc.), some are describing the process (e.g. assemble, cook, etc.).

In [55]:
# before separating amounts and units where necessary
clean_path = "recipes_cleaned.json"
with open(clean_path, 'r') as f:
    recipes = json.load(f)

# print the first few recipes
print(json.dumps(recipes[0:5], indent=4))

[
    {
        "recipe_name": "Nagi\u2019s Salad Sandwich",
        "ingredient_groups": [
            {
                "group_name": "Main",
                "ingredients": [
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "soft bread roll",
                        "notes": "wholemeal or white (or sliced bread, wraps)"
                    },
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "small carrot",
                        "notes": "peeled, grated using a box grater (3/4 cup lightly packed per sandwich)"
                    },
                    {
                        "amount": "1/2",
                        "unit": "",
                        "name": "ripe avocado",
                        "notes": ""
                    },
                    {
                        "amount": "1",
                        "unit":

Note that first recipe includes 3 \u2013 4. Need to sort that out.

Also note:
```
"group_name": "Lemon Vinaigrette",
                "ingredients": [
                    {
                        "amount": "\u00be",
                        "unit": "tsp",
                        "name": "lemon juice",
                        "notes": ""
                    },
```

In [ ]:
# separate amount and units where necessary
import re
def parse_amount(amount_str, unit_str):
    match = re.match(r'^([\d\s./]+)\s*([a-zA-Z]+)\s*/\s*([\d\s./]+)\s*([a-zA-Z]+)$', amount_str)

    # only keep the first match to keep it metric (pattern is that the first match is the one with metric units)
    if match:
        amount_str = match.group(1).strip()
        unit_str = match.group(2).strip()
        alt_amount = match.group(3).strip()
        alt_unit = match.group(4).strip()
        # print for debugging
        print(f"Amount: {amount_str}, Unit: {unit_str}, Alt Amount: {alt_amount}, Alt Unit: {alt_unit}")
    
        return amount_str, unit_str, alt_amount, alt_unit
    # if dual amount not found
    return amount_str, unit_str, None, None

with open('recipes_cleaned.json', 'r') as f:
    recipes = json.load(f)

for recipe in recipes:
    for group in recipe["ingredient_groups"]:
        for ing in group["ingredients"]:
            amount = ing["amount"]
            unit = ing["unit"]
            new_amount, new_unit, alt_amount, alt_unit = parse_amount(amount, unit)
            # set the new values
            ing["amount"] = new_amount
            ing["unit"] = new_unit

            # if alt amount and unit are set, add them to the ingredient
            if alt_amount:
                ing["alt_amount"] = alt_amount
                ing["alt_unit"] = alt_unit

with open('recipes_cleaned.json', 'w') as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

Amount: 180, Unit: g, Alt Amount: 12, Alt Unit: tbsp
Amount: 500, Unit: g, Alt Amount: 1, Alt Unit: lb
Amount: 400, Unit: g, Alt Amount: 14, Alt Unit: oz
Amount: 400, Unit: g, Alt Amount: 14, Alt Unit: oz
Amount: 120, Unit: g, Alt Amount: 4.2, Alt Unit: oz
Amount: 500, Unit: g, Alt Amount: 1, Alt Unit: lb
Amount: 200, Unit: g, Alt Amount: 7, Alt Unit: oz
Amount: 75, Unit: g, Alt Amount: 5, Alt Unit: tbsp
Amount: 200, Unit: g, Alt Amount: 7, Alt Unit: oz
Amount: 100, Unit: g, Alt Amount: 7, Alt Unit: tbsp
Amount: 250, Unit: g, Alt Amount: 8, Alt Unit: oz
Amount: 115, Unit: g, Alt Amount: 1/2, Alt Unit: cup
Amount: 200, Unit: g, Alt Amount: 7, Alt Unit: oz
Amount: 100, Unit: g, Alt Amount: 7, Alt Unit: tbsp
Amount: 250, Unit: g, Alt Amount: 8, Alt Unit: oz
Amount: 400, Unit: g, Alt Amount: 14, Alt Unit: oz
Amount: 200, Unit: g, Alt Amount: 7, Alt Unit: oz
Amount: 1.8, Unit: kg, Alt Amount: 4, Alt Unit: lb
Amount: 100, Unit: g, Alt Amount: 1, Alt Unit: stick
Amount: 750, Unit: g, Alt Amou

In [57]:
# verify 
clean_path = "recipes_cleaned.json"
with open(clean_path, 'r') as f:
    recipes = json.load(f)

# print the first few recipes
print(json.dumps(recipes[0:5], indent=4))

[
    {
        "recipe_name": "Nagi\u2019s Salad Sandwich",
        "ingredient_groups": [
            {
                "group_name": "Main",
                "ingredients": [
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "soft bread roll",
                        "notes": "wholemeal or white (or sliced bread, wraps)"
                    },
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "small carrot",
                        "notes": "peeled, grated using a box grater (3/4 cup lightly packed per sandwich)"
                    },
                    {
                        "amount": "1/2",
                        "unit": "",
                        "name": "ripe avocado",
                        "notes": ""
                    },
                    {
                        "amount": "1",
                        "unit":

In [62]:
# check if notes contain any amount / units where no amount of unit is given
for recipe in recipes:
    for group in recipe["ingredient_groups"]:
        for ing in group["ingredients"]:
            if ing["amount"] == "" and ing["unit"] == "" and ing["notes"] != "":
                print(f"  Recipe: {recipe['recipe_name']}")
                print(f"  Group:  {group['group_name']}")
                print(f"  Name:   {ing['name']}")
                print(f"  Notes:  {ing['notes']}")
                print()

  Recipe: Coconut Chicken Curry – quick and easy
  Group:  Serving
  Name:   Plain yogurt
  Notes:  slightly thinned with water to make it drizzle-able

  Recipe: Coconut Chicken Curry – quick and easy
  Group:  Serving
  Name:   Coriander leaves (cilantro)
  Notes:  roughly chopped, recommended

  Recipe: Coconut Chicken Curry – quick and easy
  Group:  Serving
  Name:   Basmati rice
  Notes:  (you'll need to cook 3 cups rice grains to make enough) or other rice of choice

  Recipe: Coconut Chicken Curry – quick and easy
  Group:  Serving
  Name:   Naan
  Notes:  optional

  Recipe: Tuna Carpaccio
  Group:  Tuna
  Name:   Few edible flowers
  Notes:  (optional)

  Recipe: 12 Minute Baked Salmon Fajitas
  Group:  Choose A Fajita Sauce
  Name:   Pink Taco Sauce
  Notes:  (2 ingredients!)

  Recipe: 12 Minute Baked Salmon Fajitas
  Group:  Serving
  Name:   Lime wedges
  Notes:  optional

  Recipe: Rainbow Chicken Stir Fry
  Group:  Serving
  Name:   White rice
  Notes:  – or any other r

Seems like the missing values are usually when it is optional, or up to user, serving, etc. May have to come back to this later.

In [ ]:
# check all unique group names in ingredient groups
group_names = set()
for r in recipes:
    for group in r["ingredient_groups"]:
        group_names.add(group["group_name"])

print(f"Total unique group names: {len(group_names)}\n")
for name in sorted(group_names):
    print(f"  • {repr(name)}")

Total unique group names: 1576

  • '"Popcorn Salt"'
  • '* Asterisk-Marked'
  • '1. Seafood Sauces For Lobster – Choose'
  • '2. Lobster Cocktail'
  • '3 Layer Vanilla Cake'
  • '3. Lobster With Herb & Garlic Butter (Enough For 2 Lobsters)'
  • '4. Lobster With French Lemon Butter Sauce (2 Lobsters)'
  • '5. Lobster With Salsa Verde (4 Lobsters)'
  • '6. Lobster Rolls'
  • 'Add Ins'
  • 'Add Ins (3 1/2 Cups Your Choice, Note 3)'
  • 'Add Ins (Mix And Match!)'
  • 'Add Ins (Note 1 For Alternatives)'
  • 'Add Ins – Choose One'
  • 'Add-Ins'
  • 'Add-Ins (7 Cups "Anything" – Note 2)'
  • 'Add-Ins (Note 7)'
  • 'Add-Ins (Or Use Your Own!)'
  • 'Add-Ins – 1 1/2 Cups “Anything You Want”, I Used'
  • 'Almond Coating (Optional, Note 5)'
  • 'Almond Decoration'
  • 'Almond Sauce'
  • 'Alternative Add In Options'
  • 'Alternative Stir In'
  • 'Amatrice Spaghetti (Spaghetti All’Amatriciana)'
  • 'Apple Cider Drizzle'
  • 'Apple Filling'
  • 'Apple Pie Filling'
  • 'Apple Pie Fries'
  • 'Apple Tu

Seems like there are primarily three groups:
- component
- serving
- instruction

In [ ]:
# keywords that suggest an instruction/action group
# got these using LLM, may not be perfect
instruction_words = [
    "assembl", "baking", "cooking", "layering", "to cook", "to bake",
    "to build", "to fry", "to make", "method", "to construct",
    "roasting", "grilling", "frying", "browning", "sautéing",
    "par boil", "velveting", "bloom", "boil and drain",
    "microwave", "oven", "stovetop", "no rinsing"
]

# collect matches
found = []
for r in recipes:
    for group in r["ingredient_groups"]:
        name = group["group_name"].lower()
        if any(word in name for word in instruction_words):
            found.append({
                "recipe": r["recipe_name"],
                "group_name": group["group_name"],
                "ingredients": group["ingredients"]
            })

# print with their ingredients
print(f"Found {len(found)} instruction-like groups:\n")

for item in found:
    print(f"Recipe: {item['recipe']}")
    print(f"Group:  {item['group_name']}")
    print(f"Ingredients ({len(item['ingredients'])}):")
    for ing in item["ingredients"]:
        print(f"  - {ing['amount']} {ing['unit']} {ing['name']}")
    print()

Found 116 instruction-like groups:

Recipe: Pastel de Choclo (Chilean Beef and Corn Casserole)
Group:  Assembling / Baking
Ingredients (5):
  - 1/4 cup pitted black olives
  - 2  soft boiled eggs
  - 2 tbsp raisins
  - 1 tbsp regular sugar (granulated sugar)
  - Pinch  paprika

Recipe: Layered Golden Rice Bake with Spiced Meat
Group:  Layering
Ingredients (2):
  - 1/4 cup pine nuts
  - 2 – 3 tbsp extra virgin olive oil

Recipe: JB’s French Pork Schnitzel
Group:  Crumbing And Cooking
Ingredients (5):
  - 1/3 cup plain flour / all-purpose flour
  - 2  eggs, lightly whisked
  - 1 1/2 cup panko breadcrumbs
  - 3 tbsp herbes de Provence
  - 1/2 cup vegetable oil

Recipe: JB’s French Pork Schnitzel
Group:  Herbes De Provence
Ingredients (4):
  - 4 tsp dried thyme
  - 3 tsp dried rosemary
  - 1 1/4 tsp dried marjoram or savory
  - 3/4 tsp dried oregano

Recipe: Panna Cotta
Group:  Bloom Gelatine
Ingredients (2):
  - 2 1/2 tsp gelatine powder
  - 1/2 cup milk

Recipe: Lemongrass Chicken Rice P

The "instructions" group still contains the ingredients. I considered that these may clash with instructions, but it is only a lack of consistency in the naming. The content of each group is still the list of ingredietns which is appropriate for the group.

Adding meta data to the ingredients however will be useful. This will help the query stage.

In [ ]:
# note that this function does some duplicate work as normalise_group_name. Can be merged later
def clean_group_name(name):
    # strip parenthetical notes, stray quotes, asterisks, and normalize
    # remove trailing parenthetical notes like (Note 1), (see notes for subs)
    name = re.sub(r'\s*\([^)]*\)\s*$', '', name)
    # remove stray leading/trailing quotes, asterisks, colons, dashes
    name = name.strip(' "\'*:-–')
    # collapse multiple spaces
    name = re.sub(r'\s+', ' ', name)
    return name.strip()

def classify_group(name):
    # assign a group_type based on the cleaned name
    name_lower = name.lower()
    
    # instruction-like
    instruction_words = [
        "assembl", "baking", "cooking", "layering", "to cook", "to bake",
        "to build", "to fry", "to make", "method", "to construct",
        "roasting", "grilling", "frying", "browning", "sautéing",
        "par boil", "velveting", "bloom", "boil and drain",
        "microwave", "oven", "stovetop", "no rinsing", "to finish",
        "to use", "to assemble", "for baking", "for cooking", "for frying",
        "for roasting", "for sautéing", "to cook & serve", "to cook / serve"
    ]
    if any(word in name_lower for word in instruction_words):
        return "instruction"
    
    # serving-like
    serving_words = [
        "serving", "garnish", "to serve", "for serving", "topping",
        "finish", "on the side", "sprinkling", "decoration",
        "for serving", "to garnish", "garnishes", "sides",
        "presentation", "toppings"
    ]
    if any(word in name_lower for word in serving_words):
        return "serving"
    
    return "component"

# load
with open("recipes_cleaned.json", "r") as f:
    recipes = json.load(f)

# process
for recipe in recipes:
    for group in recipe["ingredient_groups"]:
        original = group["group_name"]
        cleaned = clean_group_name(original)
        
        group["original_name"] = original # keep the original name, though maybe not needed
        group["group_name"] = cleaned
        group["group_type"] = classify_group(cleaned)

# save
with open("recipes_cleaned.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

# summary
types = {}
for r in recipes:
    for g in r["ingredient_groups"]:
        t = g["group_type"]
        types[t] = types.get(t, 0) + 1

print("Done. Group types summary:")
for t, count in sorted(types.items()):
    print(f"  {t}: {count}")

Done. Group types summary:
  component: 3108
  instruction: 123
  serving: 594


In [3]:
import json
clean_path = "recipes_cleaned.json"
with open(clean_path, 'r') as f:
    recipes = json.load(f)

# print the first few recipes
print(json.dumps(recipes[0:5], indent=4))

[
    {
        "recipe_name": "Nagi\u2019s Salad Sandwich",
        "ingredient_groups": [
            {
                "group_name": "Main",
                "ingredients": [
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "soft bread roll",
                        "notes": "wholemeal or white (or sliced bread, wraps)"
                    },
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "small carrot",
                        "notes": "peeled, grated using a box grater (3/4 cup lightly packed per sandwich)"
                    },
                    {
                        "amount": "1/2",
                        "unit": "",
                        "name": "ripe avocado",
                        "notes": ""
                    },
                    {
                        "amount": "1",
                        "unit":

In regards to cleaning the instructions field...
The model needs to be able to identify what the instruction group is aiming to achieve:
- so is it a component it is trying to make.

It seems like the ABBREVIATED RECIPE is a quick version of the entire recipe. Given this is true, this information may not be needed and can be discarded.

Another point of interest is that the author sometimes states "I like to..." which suggests she is recommending a step. I suppose this should be interpretted as do it.

In [4]:
# collect all unique instruction group names
inst_group_names = set()
for r in recipes:
    for group in r["instruction_groups"]:
        inst_group_names.add(group["group_name"])

print(f"Total unique instruction group names: {len(inst_group_names)}\n")
for name in sorted(inst_group_names, key=lambda x: (x is None, str(x))):
    print(f"  • {repr(name)}")

Total unique instruction group names: 1239

  • ''
  • '!Mustard Sauce'
  • '"Holiday Tapenade":'
  • '"Popcorn Salt"'
  • '(ABBREVIATED RECIPE)'
  • '(FULL RECIPE)'
  • '.'
  • '3 Minute Festive “Baked” Brie:'
  • '<br>'
  • 'ABBREVIATED'
  • 'ABBREVIATED RECIPE'
  • 'ABBREVIATED RECIPE:'
  • 'ABBREVIATED:'
  • 'AMATRICE SPAGHETTI (SPAGHETTI ALL’AMATRICIANA):'
  • 'ASSEMBLE:'
  • 'ASSEMBLING:'
  • 'ASSEMBLY & BAKING:'
  • 'Abbreviated Recipe'
  • 'Abbreviated directions:'
  • 'Abbreviated recipe'
  • 'Abbreviated recipe (hand-mixed):'
  • 'Abbreviated recipe:'
  • 'Abbreviated steps for baking pros:'
  • 'Additional Flavouring Suggestions'
  • 'Aioli:'
  • 'Alfredo Sauce:'
  • 'All Dressings except Caesar:'
  • 'All sauces except Thai Sauce:'
  • 'Almond Sauce'
  • 'Almond bits:'
  • 'Amount to Use (Note 6):'
  • 'Apple Cider Drizzle'
  • 'Apple Pie Filling:'
  • 'Apple:'
  • 'Apples:'
  • 'Aromatics'
  • 'Assemble'
  • 'Assemble & Bake'
  • 'Assemble & Bake:'
  • 'Assemble & bake:'
 

Couple of things to clean up here:
- Normalise capitalisation
- Replace None, '.', ''. '<br>'
- strip excess characters, e.g. brackets, quotation marks, etc
- strip "see note 1" etc
- some names include "3 Minute... ", strip that and keep the name only.

These don't add meaningful information.

Now, it would be ideal to have a type that labels the instruction group as sub recipe. If this is properly implemented, then components and action type in ingredients should be replaced to match sub recipe.

In [8]:
# investigate if what I labelled for group_type in ingredients as 
# component and instructions are effectively the same




In [9]:
import json

with open("recipes_cleaned.json", "r") as f:
    recipes = json.load(f)

# Find recipes that have "instruction" or "component" type groups of interest
instruction_recipes = []
component_recipes = []

for r in recipes:
    for group in r["ingredient_groups"]:
        if group["group_type"] == "instruction" and len(instruction_recipes) < 3:
            instruction_recipes.append(r)
            break
        elif group["group_type"] == "component" and len(component_recipes) < 3:
            component_recipes.append(r)
            break

# Print full recipes
def print_full_recipe(r):
    print(f"\n{'═' * 70}")
    print(f"  RECIPE: {r['recipe_name']}")
    print(f"  Servings: {r['servings']}")
    print(f"{'═' * 70}")
    
    print(f"\n  INGREDIENT GROUPS:")
    for group in r["ingredient_groups"]:
        print(f"  ┌─ [{group['group_name']}]  type: {group['group_type']}")
        for ing in group["ingredients"]:
            print(f"  │  {{")
            print(f"  │    \"amount\": {json.dumps(ing['amount'])},")
            print(f"  │    \"unit\": {json.dumps(ing['unit'])},")
            print(f"  │    \"name\": {json.dumps(ing['name'])},")
            print(f"  │    \"notes\": {json.dumps(ing['notes'])}")
            if "alt_amount" in ing:
                print(f"  │    \"alt_amount\": {json.dumps(ing['alt_amount'])},")
                print(f"  │    \"alt_unit\": {json.dumps(ing['alt_unit'])}")
            print(f"  │  }},")
        print(f"  └─")
    
    print(f"\n  INSTRUCTION GROUPS:")
    for group in r["instruction_groups"]:
        print(f"  ┌─ [{group['group_name']}]")
        for step in group["instructions"]:
            print(f"  │  \"{step}\",")
        print(f"  └─")
    
    if r.get("notes"):
        print(f"\n  NOTES:")
        for note in r["notes"]:
            print(f"  • {note}")

print("\n\n═══════════════════════════════════════")
print("RECIPES WITH 'INSTRUCTION' TYPE GROUPS")
print("═══════════════════════════════════════")
for r in instruction_recipes:
    print_full_recipe(r)

print("\n\n═══════════════════════════════════════")
print("RECIPES WITH 'COMPONENT' TYPE GROUPS")
print("═══════════════════════════════════════")
for r in component_recipes:
    print_full_recipe(r)



═══════════════════════════════════════
RECIPES WITH 'INSTRUCTION' TYPE GROUPS
═══════════════════════════════════════

══════════════════════════════════════════════════════════════════════
  RECIPE: Pastel de Choclo (Chilean Beef and Corn Casserole)
  Servings: 8.0
══════════════════════════════════════════════════════════════════════

  INGREDIENT GROUPS:
  ┌─ [Beef Filling]  type: component
  │  {
  │    "amount": "2",
  │    "unit": "tbsp",
  │    "name": "olive oil",
  │    "notes": ""
  │  },
  │  {
  │    "amount": "1",
  │    "unit": "kg",
  │    "name": "beef mince",
  │    "notes": "(ground beef)"
  │    "alt_amount": "2",
  │    "alt_unit": "lb"
  │  },
  │  {
  │    "amount": "3",
  │    "unit": "",
  │    "name": "large onions",
  │    "notes": "diced (4 \u2013 5 medium ones)"
  │  },
  │  {
  │    "amount": "6",
  │    "unit": "",
  │    "name": "large garlic cloves",
  │    "notes": "finely minced"
  │  },
  │  {
  │    "amount": "1/4",
  │    "unit": "cup",
  │    "n

The instruction is not quite a sub recipe. 

None cannot be replaced to "Main". This is not true in all cases.

Instead of focusing on the group type

Focus on matching the headers of ingredient groups and instruction groups. Given these are broadly consistent, then you can treat inconsistent values are missing values. 

This approach is far less "let's hope the datasets were aligned properly".